In [ ]:
%matplotlib qt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mne, os, pickle, csv, json
from pyxdf import load_xdf
import pandas as pd
from scipy.stats import ttest_ind, ttest_1samp

In [ ]:
# Create output folder
out_dir = 'alpha_results'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [ ]:
# Input files
input_dir = r'S:\laura-wheeler_in-ear-eeg-auditory-bci_0589_data_prism\Raw Data\Study Data Organized - in-ear EEG and scalp working'

# Find all xdf files
xdf_files = {}
participant_folders = [f for f in os.listdir(input_dir) if f.startswith('Participant') and f[-1].isdigit()]
for participant_folder in participant_folders:
    xdf_files[participant_folder] = [f for f in os.listdir(os.path.join(input_dir, participant_folder, 'sourcedata')) if 'eyes' in f and f.endswith('.xdf')]

xdf_files

In [ ]:
# Parameters
epoch_twin_size = 5

# Load EEG
epochs = {}
for participant in xdf_files:
    print('Parasing XDF files for %s' % participant)
    
    streams = {}
    for f in range(len(xdf_files[participant])):
        xdf_file = xdf_files[participant][f]
        print('file: ', xdf_file)
        
        # Load LSL streams from xdf
        found_streams, _ = load_xdf(os.path.join(input_dir, participant, 'sourcedata', xdf_file))
        for found_stream in found_streams:
            stream_name = found_stream['info']['name'][0]
            if stream_name not in streams:
                streams[stream_name] = []
            streams[stream_name].append(found_stream)
            
    # Create Raw data
    eeg_raw_list, raw = {}, {}
    for stream in streams:
        # Ignore marker stream
        if stream == 'mindset_Marker_PRISM-DIY-MSI-4':
            continue

        # Load EEG data
        for i in range(len(streams[stream])):
            print('Processing %s stream' % stream)
            # Get EEG info
            if stream == 'BrainAmpSeries-Dev_1':
                sfreq = float(streams[stream][i]['info']['nominal_srate'][0])
                ch_names = [ch['label'][0] for ch in streams[stream][i]['info']['desc'][0]['channels'][0]['channel']]
            elif stream == 'InEarEEG':
                sfreq = float(streams[stream][i]['info']['effective_srate'])
                ch_names = ['A1', 'A2']
            eeg_info = mne.create_info(ch_names, sfreq, ch_types='eeg')
            montage = mne.channels.read_custom_montage('%s_montage.elc' % stream)

            # Apply scaling factor to EEG data
            if stream == 'InEarEEG':
                eeg_data = np.transpose(streams[stream][i]['time_series']) / 24 / 4.1887e9 * 4.5
            elif stream == 'BrainAmpSeries-Dev_1':
                eeg_data = np.transpose(streams[stream][i]['time_series']) / 1e6
            eeg_raw = mne.io.RawArray(eeg_data, eeg_info, verbose=False)
            eeg_raw.set_montage(montage)

            # marker annotations
            onset, duration, description = [], [], []
            markers = streams['mindset_Marker_PRISM-DIY-MSI-4'][i]['time_series']
            markers_ts = streams['mindset_Marker_PRISM-DIY-MSI-4'][i]['time_stamps']
            for m in range(len(markers)):
                marker = json.loads(markers[m][0])['status']
                if 'eyes' in marker:
                    # Extract overlapping 5s epochs with 1s stride from 15s trials
                    start = np.arange(0, 15-epoch_twin_size+1, epoch_twin_size) + markers_ts[m] - streams[stream][i]['time_stamps'][0]
                    for s in start:
                        onset.append(s)
                        duration.append(epoch_twin_size)
                        description.append(marker)
            annotations = mne.Annotations(onset, duration, description)
            eeg_raw = eeg_raw.set_annotations(annotations)
            if stream == 'InEarEEG':
                eeg_raw = eeg_raw.resample(250)

            # Add raw to list
            if stream not in eeg_raw_list:
                eeg_raw_list[stream] = []
            eeg_raw_list[stream].append(eeg_raw)

        # Concatenate all epochs
        raw[stream] = mne.concatenate_raws(eeg_raw_list[stream])
        raw[stream] = raw[stream].filter(l_freq=1.0, h_freq=40.0)

        # Create epochs
        events, event_id = mne.events_from_annotations(raw[stream])
        if participant not in epochs:
            epochs[participant] = {}
        epochs[participant][stream] = mne.Epochs(
            raw[stream], events, event_id=event_id, preload=True,
            tmin=0., tmax=epoch_twin_size, baseline=None
        )

In [ ]:
# Compute PSD
psd_method = 'multitaper'
psds = {}

plt.close('all')
for participant in epochs:
    if participant not in psds:
        psds[participant] = {}
    for headset in epochs[participant]:
        psds[participant][headset] = epochs[participant][headset].compute_psd(method=psd_method, fmin=1, fmax=40, normalization='full')

psds

In [ ]:
# Plot PSD
plot_average = False
dB = True

plt.close('all')
for i in range(len(epochs)):
    participant = list(epochs.keys())[i]
    
    fig, ax = plt.subplots(3,2, figsize=(10,15))
    title = 'Participant %d Alpha Wave Modulation Power Spectral Density' % (i+1)
    fig.suptitle(title)

    psds[participant]['BrainAmpSeries-Dev_1']['eyes open'].plot(axes=ax[0,0], average=plot_average, dB=dB)
    psds[participant]['BrainAmpSeries-Dev_1']['eyes closed'].plot(axes=ax[0,1], average=plot_average, dB=dB)
    psds[participant]['BrainAmpSeries-Dev_1']['eyes open'].pick(['T7','T8']).plot(axes=ax[1,0], average=plot_average, dB=dB)
    psds[participant]['BrainAmpSeries-Dev_1']['eyes closed'].pick(['T7','T8']).plot(axes=ax[1,1], average=plot_average, dB=dB)
    psds[participant]['InEarEEG']['eyes open'].plot(axes=ax[2,0], average=plot_average, dB=dB)
    psds[participant]['InEarEEG']['eyes closed'].plot(axes=ax[2,1], average=plot_average, dB=dB)

    ax[0,0].set_xlabel('')
    ax[0,1].set_xlabel('')
    ax[1,0].set_xlabel('')
    ax[1,1].set_xlabel('')
    ax[2,0].set_xlabel('Frequency (Hz)')
    ax[2,1].set_xlabel('Frequency (Hz)')

    ax[0,0].set_title('Eyes Open')
    ax[0,1].set_title('Eyes Closed')
    ax[1,0].set_title('')
    ax[1,1].set_title('')
    ax[2,0].set_title('')
    ax[2,1].set_title('')

    ax[0,0].set_ylabel('Scalp EEG\nPower (dB µ$V^{2}$/Hz')
    ax[0,1].set_ylabel('')
    ax[1,0].set_ylabel('Temporal EEG\nPower (dB µ$V^{2}$/Hz')
    ax[1,1].set_ylabel('')
    ax[2,0].set_ylabel('In-Ear EEG\nPower (dB µ$V^{2}$/Hz')
    ax[2,1].set_ylabel('')

    scalp_ylim = [min(ax[0,0].get_ylim()[0], ax[0,1].get_ylim()[0]), max(ax[0,0].get_ylim()[1], ax[0,1].get_ylim()[1])]
    temporal_ylim = [min(ax[1,0].get_ylim()[0], ax[1,1].get_ylim()[0]), max(ax[1,0].get_ylim()[1], ax[1,1].get_ylim()[1])]
    inear_ylim = [min(ax[2,0].get_ylim()[0], ax[2,1].get_ylim()[0]), max(ax[2,0].get_ylim()[1], ax[2,1].get_ylim()[1])]

    ax[0,0].set_ylim(scalp_ylim)
    ax[0,1].set_ylim(scalp_ylim)
    ax[1,0].set_ylim(temporal_ylim)
    ax[1,1].set_ylim(temporal_ylim)
    ax[2,0].set_ylim(inear_ylim)
    ax[2,1].set_ylim(inear_ylim)

    for a in [ax[1,0], ax[1,1]]:
        for line in a.get_lines():
            # print(line.get_color())
            if (line.get_color() == np.array([[0., 0., 1.]])).all():
                line.set_color(np.array([0., 0.549, 0.]))
            if (line.get_color() == np.array([1., 1., 0.])).all():
                line.set_color(np.array([1., 0.5529, 0.]))
    for head in [fig.axes[8], fig.axes[9]]:
        head.get_children()[0].set_color(np.array([[0., 0.549, 0., 1.], [1., 0.5529, 0., 1.]]))

    plt.tight_layout()
    plt.show()

    # Save PSD plot
    plt.savefig(os.path.join(out_dir, '%s.png' % title))
    # break

In [ ]:
# Compute peak alpha modulation ratio (RAM)
def compute_ram(power_closed, power_open):
    power_open = np.mean(np.mean(np.max(power_open, axis=-1), axis=-1))
    power_closed = np.mean(np.max(power_closed, axis=-1), axis=-1)
    
    return np.mean(power_closed / power_open)

fmin, fmax = 7, 15
scalings = mne.defaults._handle_default("scalings", None)['eeg']
power = {}
ram = []
for i in range(len(epochs)):
    participant = list(epochs.keys())[i]
    power[participant] = {'scalp': {}, 'temporal': {}, 'ear': {}}
    for task in ['eyes open', 'eyes closed']:
        # get PSD in frequency range
        power[participant]['scalp'][task] = psds[participant]['BrainAmpSeries-Dev_1'][task].get_data(fmin=fmin, fmax=fmax)
        power[participant]['temporal'][task] = psds[participant]['BrainAmpSeries-Dev_1'][task].get_data(picks=['T7','T8'], fmin=fmin, fmax=fmax)
        power[participant]['ear'][task] = psds[participant]['InEarEEG'][task].get_data(fmin=fmin, fmax=fmax)

        # scale power to dB
        power[participant]['scalp'][task] = power[participant]['scalp'][task] * scalings * scalings
        power[participant]['temporal'][task] = power[participant]['temporal'][task] * scalings * scalings
        power[participant]['ear'][task] = power[participant]['ear'][task] * scalings * scalings

    # compute RAM
    ram.append({
        'Participant': int(participant.replace('Participant ','')),
        'Scalp RAM': compute_ram(power[participant]['scalp']['eyes closed'], power[participant]['scalp']['eyes open']),
        'Temporal RAM': compute_ram(power[participant]['temporal']['eyes closed'], power[participant]['temporal']['eyes open']),
        'In-Ear RAM': compute_ram(power[participant]['ear']['eyes closed'], power[participant]['ear']['eyes open']),
    })

# Save and print results
ram = pd.DataFrame(ram)
ram.loc[len(ram)] = ['average', ram['Scalp RAM'].mean(), ram['Temporal RAM'].mean(), ram['In-Ear RAM'].mean()]
ram = ram.round(decimals=2)
ram_fname = os.path.join(out_dir, 'alpha_modulation_ratio.csv')
ram.to_csv(ram_fname)
print('Saved results to: %s' % ram_fname)
ram

In [ ]:
# T-test to test if closed condition is significantly higher than open
def perform_test(power_closed, power_open):
    # power_closed = np.mean(np.max(power_closed, axis=-1), axis=-1)
    # power_open = np.mean(np.max(power_open, axis=-1), axis=-1)
    # result = ttest_ind(power_closed, power_open, alternative='greater', equal_var=False, permutations=100000)
    # return np.mean(result.pvalue)

    power_open = np.mean(np.mean(np.max(power_open, axis=-1), axis=-1))
    power_closed = np.mean(np.max(power_closed, axis=-1), axis=-1)
    power_closed = power_closed / power_open
    result = ttest_1samp(power_closed, 1, alternative='greater')
    return result.pvalue

ttest_results = []
for participant in power:
    ttest_results.append({
        'Participant': participant.replace('Participant ',''),
        'Scalp p-value': perform_test(power[participant]['scalp']['eyes closed'], power[participant]['scalp']['eyes open']),
        'Temporal p-value': perform_test(power[participant]['temporal']['eyes closed'], power[participant]['temporal']['eyes open']),
        'In-Ear p-value': perform_test(power[participant]['ear']['eyes closed'], power[participant]['ear']['eyes open'])
    })

# Compute avg stats
avg_power = {'scalp':{}, 'temporal': {}, 'ear': {}}
for participant in power:
    for headset in avg_power:
        for task in ['eyes closed', 'eyes open']:
            if task not in avg_power[headset]:
                avg_power[headset][task] = []
            avg_power[headset][task].append(power[participant][headset][task])
for headset in avg_power:
    for task in ['eyes closed', 'eyes open']:
        avg_power[headset][task] = np.concatenate(avg_power[headset][task], axis=0)

ttest_results.append({
    'Participant': 'average',
    'Scalp p-value': perform_test(avg_power['scalp']['eyes closed'], avg_power['scalp']['eyes open']),
    'Temporal p-value': perform_test(avg_power['temporal']['eyes closed'], avg_power['temporal']['eyes open']),
    'In-Ear p-value': perform_test(avg_power['ear']['eyes closed'], avg_power['ear']['eyes open'])
})

# Save stats
ttest_results = pd.DataFrame(ttest_results)
ttest_fname = os.path.join(out_dir, 'alpha_ttest_results.csv')
ttest_results.to_csv(ttest_fname)
print('Saved results to: %s' % ttest_fname)
ttest_results

In [ ]:
# Plot scalp, temporal, ear power for each participant
df = []
names = {'scalp': 'Scalp', 'temporal': 'Temporal', 'ear': 'In-Ear'}
for participant in power:
    row = {'Participant': participant}
    for headset in ['scalp', 'temporal', 'ear']:
        power_closed = power[participant][headset]['eyes closed']
        power_open = power[participant][headset]['eyes open']
        power_open = np.mean(np.mean(np.max(power_open, axis=-1), axis=-1))
        power_closed = np.mean(np.max(power_closed, axis=-1), axis=-1)
        power_closed = power_closed / power_open
        row[names[headset]] = np.mean(power_closed)
        row['%s Error' % (names[headset])] = np.std(power_closed) / np.sqrt(power_closed.size)
    df.append(row)
df = pd.DataFrame(df)

plt.close('all')
ax = df[['Scalp', 'Temporal', 'In-Ear']].plot.bar(rot=0)
plt.xlabel('Participant')
plt.ylabel('Alpha Modulation Ratio')
plt.xticks([0, 1, 2, 3, 4], [1, 2, 3, 4, 5])

errors = np.ravel(df[['Scalp Error', 'Temporal Error', 'In-Ear Error']].values.T)
for patch, moe in zip(ax.patches, errors):
    height = patch.get_height() # of bar
    min_y, max_y = height - moe, height + moe
    plt.vlines(patch.get_x() + patch.get_width()/2, min_y, max_y, color='k')

plt.ylim([0, None])
plt.show()
plt.savefig(os.path.join(out_dir, 'alpha_modulation_ratio_by_participant.png'))

In [ ]:
df